# 1.Setup

In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet(r'../data/interim/pages_extracted.parquet')

# 2.EDA

In [3]:
df.head()

,trimestre,ano,tipo_doc,page_number,page_count,char_count,text
0,1,2024,completas,1,147,126,1T24 \n\nItaú Unibanco Holding S.A. \n\n**Anál...
1,1,2024,completas,2,147,897,#### **Índice** \n\n###### **Análise gerencial...
2,1,2024,completas,5,147,3851,###### Sumário do **Resultado Gerencial** \n\n...
3,1,2024,completas,6,147,3068,###### Resultado **Gerencial** \n\nNesse relat...
4,1,2024,completas,7,147,4437,###### Demonstração de **Resultado do 1o trime...


In [4]:
df.tail()

,trimestre,ano,tipo_doc,page_number,page_count,char_count,text
1478,4,2025,completas,165,169,3329,Itaú Unibanco Holding S.A. \n\n###### **Outras...
1479,4,2025,completas,166,169,3328,Itaú Unibanco Holding S.A. \n\nComo parte de u...
1480,4,2025,completas,167,169,915,Itaú Unibanco Holding S.A. \n\nDos assuntos qu...
1481,4,2025,completas,168,169,841,##### **ITAÚ UNIBANCO HOLDING S.A.** \n\nCNPJ ...
1482,4,2025,completas,169,169,1840,###### **ITAÚ UNIBANCO HOLDING S.A.** \n\n####...


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1483 entries, 0 to 1482
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   trimestre    1483 non-null   int64
 1   ano          1483 non-null   int64
 2   tipo_doc     1483 non-null   str  
 3   page_number  1483 non-null   int64
 4   page_count   1483 non-null   int64
 5   char_count   1483 non-null   int64
 6   text         1483 non-null   str  
dtypes: int64(5), str(2)
memory usage: 4.5 MB


In [6]:
# checando se todos os 10 documentos estao presentes e
# outros detalhes
resumo = df.groupby(["ano", "trimestre", "tipo_doc"])["page_number"].agg(
    total_paginas_uteis="count",
    primeira_pagina="min",
    ultima_pagina="max"
).reset_index()

resumo

,ano,trimestre,tipo_doc,total_paginas_uteis,primeira_pagina,ultima_pagina
0,2024,1,completas,140,1,147
1,2024,2,completas,150,1,157
2,2024,3,completas,139,1,149
3,2024,4,completas,150,1,164
4,2025,1,condensadas,145,1,153
5,2025,2,condensadas,154,1,163
6,2025,3,condensadas,144,1,153
7,2025,4,completas,161,1,169
8,2026,1,condensadas,146,1,153
9,2026,2,completas,154,1,162


In [7]:
# checando dist de caracteres
df["char_count"].describe()

count    1483.000000
mean     3033.331760
std      1269.430866
min        88.000000
25%      2152.000000
50%      2999.000000
75%      4018.500000
max      6444.000000
Name: char_count, dtype: float64

In [8]:
# checando total de caracteres
df["char_count"].sum()

np.int64(4498431)

In [9]:
# inspecionando as 20 com menos qtd de caracteres
paginas_curtas = df.nsmallest(20, "char_count")

for _, row in paginas_curtas.iterrows():
    print("="*40, f"{row['trimestre']}T{row['ano']} | Pág {row['page_number']}", "="*40)
    print(f" ----- ({row['char_count']} chars)")
    print('Conteudo da pagina: \n')
    print(row["text"])

======================================== 4T2024 | Pág 3 ========================================
 ----- (88 chars)
Conteudo da pagina: 

# **Análise Gerencial da Operação** 

4 o Trimestre de 2024 

Itaú Unibanco Holding S.A.
======================================== 2T2026 | Pág 35 ========================================
 ----- (91 chars)
Conteudo da pagina: 

#### **Demonstrações Contábeis** Completas Individuais e Consolidadas 

30 de junho de 2026
======================================== 1T2025 | Pág 1 ========================================
 ----- (94 chars)
Conteudo da pagina: 

**Análise Gerencial** da Operação e Demonstrações Contábeis Condensadas 

1o trimestre de 2025
======================================== 2T2025 | Pág 1 ========================================
 ----- (94 chars)
Conteudo da pagina: 

**Análise Gerencial da Operação** e Demonstrações Contábeis Condensadas 

2o Trimestre de 2025
======================================== 3T2025 | Pág 1 ========================

In [ ]:
# verificando quantas paginas tem menos de 200 char:
print(f"Páginas com menos de 150 caracteres: {len(df[df['char_count'] < 150])}")
print(f"Páginas com menos de 200 caracteres: {len(df[df['char_count'] < 200])}")

Páginas com menos de 150 caracteres: 22
Páginas com menos de 200 caracteres: 24


In [25]:
# inspecionando todas as paginas que seriam excluidas
limite_chars = 200
paginas_excluidas = df[df["char_count"] < limite_chars].sort_values("char_count")

print(f"Total de páginas a serem excluídas (< {limite_chars} chars): {len(paginas_excluidas)}\n")

# exibe os metadados e o conteúdo textual exato de cada uma
for i, (_, row) in enumerate(paginas_excluidas.iterrows(), start=1):
    print(f"{i:02d}. {row['trimestre']}T{row['ano']} | Pág {row['page_number']} ({row['char_count']} chars) | Tipo: {row['tipo_doc']}")
    print(f"Texto: {repr(row['text'])}")
    print("-" * 80)

# visao resumida por número da página no PDF
print("\nContagem por número da página:")
print(paginas_excluidas["page_number"].value_counts().sort_index())

Total de páginas a serem excluídas (< 200 chars): 24

01. 4T2024 | Pág 3 (88 chars) | Tipo: completas
Texto: '# **Análise Gerencial da Operação** \n\n4 o Trimestre de 2024 \n\nItaú Unibanco Holding S.A.'
--------------------------------------------------------------------------------
02. 2T2026 | Pág 35 (91 chars) | Tipo: completas
Texto: '#### **Demonstrações Contábeis** Completas Individuais e Consolidadas \n\n30 de junho de 2026'
--------------------------------------------------------------------------------
03. 2T2025 | Pág 1 (94 chars) | Tipo: condensadas
Texto: '**Análise Gerencial da Operação** e Demonstrações Contábeis Condensadas \n\n2o Trimestre de 2025'
--------------------------------------------------------------------------------
04. 1T2025 | Pág 1 (94 chars) | Tipo: condensadas
Texto: '**Análise Gerencial** da Operação e Demonstrações Contábeis Condensadas \n\n1o trimestre de 2025'
--------------------------------------------------------------------------------
05. 3T20

Então, basicamente, todas as páginas com menos de 200 caracteres são páginas de transição. Portanto, podem e devem ser excluidas da nossa base. Entretanto, o item 24 da análise se provou ser uma página que não de transição, portanto, irei usar o filtro de 150 caracteres.

O filtro final irá excluir a primeira página e páginas com menos de 150 caracteres.

In [ ]:
# aplicando o filtro
df_clean = df[(df["page_number"] > 1) & (df["char_count"] >= 150)]

,trimestre,ano,tipo_doc,page_number,page_count,char_count,text
1,1,2024,completas,2,147,897,#### **Índice** \n\n###### **Análise gerencial...
2,1,2024,completas,5,147,3851,###### Sumário do **Resultado Gerencial** \n\n...
3,1,2024,completas,6,147,3068,###### Resultado **Gerencial** \n\nNesse relat...
4,1,2024,completas,7,147,4437,###### Demonstração de **Resultado do 1o trime...
5,1,2024,completas,8,147,5735,###### Análise do desempenho do 1o trimestre d...


In [23]:
df_clean.loc[df_clean.page_number == df_clean.page_count]

,trimestre,ano,tipo_doc,page_number,page_count,char_count,text
139,1,2024,completas,147,147,1610,###### **ITAÚ UNIBANCO HOLDING S.A.** \n\n####...
284,1,2025,condensadas,153,153,1593,###### **ITAÚ UNIBANCO HOLDING S.A.** \n\n####...
430,1,2026,condensadas,153,153,1799,###### **ITAÚ UNIBANCO HOLDING S.A.** \n\n**CN...
580,2,2024,completas,157,157,1647,###### **ITAÚ UNIBANCO HOLDING S.A.** \n\n####...
734,2,2025,condensadas,163,163,1639,###### **ITAÚ UNIBANCO HOLDING S.A.** \n\n####...
888,2,2026,completas,162,162,1840,###### **ITAÚ UNIBANCO HOLDING S.A.** \n\n####...
1027,3,2024,completas,149,149,1591,###### **ITAÚ UNIBANCO HOLDING S.A.** \n\n####...
1171,3,2025,condensadas,153,153,1762,"**Praça Alfredo Egydio de Souza Aranha no 100,..."
1321,4,2024,completas,164,164,1639,###### **ITAÚ UNIBANCO HOLDING S.A.** \n\n####...
1482,4,2025,completas,169,169,1840,###### **ITAÚ UNIBANCO HOLDING S.A.** \n\n####...


Com isso, percebi que todas as últimas páginas de todos os documentos foram removidas.